# 06 — Multiple Testing Correction

Every time we run a hypothesis test at α=0.05, we accept a 5% false positive rate.
If we run **k** tests, the probability of at least one false positive grows rapidly.

This notebook covers:

1. **The inflation problem** — simulating false positives under the null
2. **Family-wise error rate (FWER)** — the probability of any false positive
3. **Bonferroni correction** — the simplest FWER control
4. **Holm-Bonferroni** — a uniformly more powerful stepwise alternative
5. **Benjamini-Hochberg FDR** — controlling the *rate* of false positives
6. **Practical guidance** — when to use each method

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append("../src")
from ab_stats import bonferroni_correct, benjamini_hochberg

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

np.random.seed(42)
ALPHA = 0.05

## 1. The Inflation Problem

Suppose we test 20 metrics, none of which are truly affected by the treatment.
Under the null hypothesis, each test has a 5% chance of a false positive.

$$P(\text{at least one false positive}) = 1 - (1 - 0.05)^{20} \approx 64\%$$

That's not a p-value problem — it's a *multiple comparisons* problem.

In [ ]:
# Simulate 10,000 experiments, each testing k null metrics
k_values = [1, 5, 10, 20, 50, 100]
fwer = [1 - (1 - ALPHA)**k for k in k_values]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, [f*100 for f in fwer], marker="o", color="tomato", linewidth=2)
ax.axhline(5, color="steelblue", linestyle="--", linewidth=1.2, label="Nominal α=5%")
ax.set_xlabel("Number of tests (k)")
ax.set_ylabel("P(at least one false positive) %")
ax.set_title("Family-wise error rate grows with number of tests")
ax.legend()
for k, f in zip(k_values, fwer):
    ax.annotate(f"{f:.0%}", (k, f*100), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Simulate false positives empirically
n_sims = 10_000
k = 20
n_per_group = 5_000

false_pos_counts = []
for _ in range(n_sims):
    p_values = []
    for _ in range(k):
        g1 = np.random.binomial(1, 0.12, n_per_group)
        g2 = np.random.binomial(1, 0.12, n_per_group)  # null: same rate
        _, pv = stats.ttest_ind(g1, g2)
        p_values.append(pv)
    false_pos_counts.append(sum(p < ALPHA for p in p_values))

any_false_pos = np.mean([c > 0 for c in false_pos_counts])
print(f"Simulated FWER with k={k} tests: {any_false_pos:.2%}")
print(f"Theoretical FWER:               {1-(1-ALPHA)**k:.2%}")

## 2. Bonferroni Correction

The simplest approach: divide α by the number of tests.

$$\alpha_{\text{adjusted}} = \frac{\alpha}{k}$$

A test is significant only if its p-value is below the adjusted threshold.

**Pros:** Simple, controls FWER strongly
**Cons:** Very conservative — inflates Type II errors (misses real effects)
when tests are positively correlated

In [ ]:
# Apply to our segment p-values from notebook 05
segment_results = {
    "Weekday":   0.312,
    "Weekend":   0.421,
    "Monday":    0.189,
    "Tuesday":   0.743,
    "Wednesday": 0.051,
    "Thursday":  0.088,
    "Friday":    0.034,   # looks significant unadjusted!
    "Saturday":  0.291,
    "Sunday":    0.578,
}

names   = list(segment_results.keys())
p_vals  = np.array(list(segment_results.values()))

adj_alpha, bonf_rejected, bonf_p = bonferroni_correct(p_vals, ALPHA)
print(f"Bonferroni adjusted alpha: {adj_alpha:.4f}")
print()
print("{:<12} {:>10} {:>10} {:>10}".format("Segment", "p-value", "Bonf p", "Rejected"))
print("-" * 46)
for name, p, bp, rej in zip(names, p_vals, bonf_p, bonf_rejected):
    flag = "* SIGNIFICANT" if rej else ""
    print(f"{name:<12} {p:>10.4f} {min(bp,1):>10.4f} {str(rej):>10} {flag}")

## 3. Holm-Bonferroni (Stepwise)

Sort p-values from smallest to largest. Compare each to α/(k−rank+1) instead of α/k.
Stop as soon as a test is not rejected — all remaining are also not rejected.

This is uniformly more powerful than Bonferroni (never worse, often better)
while still controlling FWER.

In [ ]:
order = np.argsort(p_vals)
k = len(p_vals)
holm_rejected = np.zeros(k, dtype=bool)

print("{:<6} {:<12} {:>10} {:>12} {:>10}".format("Rank", "Segment", "p-value", "Threshold", "Rejected"))
print("-" * 54)

stop = False
for rank, idx in enumerate(order):#, 1):
    threshold = ALPHA / (k - rank)
    if not stop and p_vals[idx] < threshold:
        holm_rejected[idx] = True
    else:
        stop = True
    print(f"{rank+1:<6} {names[idx]:<12} {p_vals[idx]:>10.4f} {threshold:>12.4f} {str(holm_rejected[idx]):>10}")

print()
print(f"Bonferroni rejections: {bonf_rejected.sum()} | Holm rejections: {holm_rejected.sum()}")

## 4. Benjamini-Hochberg FDR

Instead of controlling **FWER** (any false positive), BH controls the
**False Discovery Rate** — the *expected proportion* of rejections that are false.

$$FDR = E\left[\frac{\text{false positives}}{\text{total rejections}}\right]$$

This is less stringent than Bonferroni — appropriate when you are OK with
some false positives as long as most findings are real.

**Use BH when:** exploring many features, genes, or segments in an discovery context
**Use Bonferroni/Holm when:** confirmatory testing where any false positive is costly

In [ ]:
bh_rejected, bh_p = benjamini_hochberg(p_vals, ALPHA)

print("{:<12} {:>10} {:>10} {:>10}".format("Segment", "p-value", "BH adj p", "Rejected"))
print("-" * 46)
for name, p, bp, rej in zip(names, p_vals, bh_p, bh_rejected):
    flag = "* SIGNIFICANT" if rej else ""
    print(f"{name:<12} {p:>10.4f} {bp:>10.4f} {str(rej):>10} {flag}")

In [ ]:
# Visual comparison of all three methods
fig, ax = plt.subplots(figsize=(10, 5))

y = np.arange(len(names))
order_display = np.argsort(p_vals)
sorted_names = [names[i] for i in order_display]
sorted_p     = p_vals[order_display]
sorted_bonf  = bonf_rejected[order_display]
sorted_holm  = holm_rejected[order_display]
sorted_bh    = bh_rejected[order_display]

ax.barh(y, sorted_p, color="lightgray", edgecolor="white")
ax.axvline(ALPHA,              color="gray",     linestyle=":",  linewidth=1.5, label=f"Unadjusted α={ALPHA}")
ax.axvline(adj_alpha,          color="tomato",   linestyle="--", linewidth=1.5, label=f"Bonferroni α={adj_alpha:.4f}")

for i, (bp, hp, bhp) in enumerate(zip(sorted_bonf, sorted_holm, sorted_bh)):
    markers = []
    if bhp:   markers.append(("BH",       "green",     0.82))
    if hp:    markers.append(("Holm",     "orange",    0.88))
    if bp:    markers.append(("Bonf",     "tomato",    0.94))
    for label, color, xpos in markers:
        ax.text(xpos, i, label, color=color, fontsize=7, va="center")

ax.set_yticks(y)
ax.set_yticklabels(sorted_names)
ax.set_xlabel("p-value")
ax.set_title("Segment p-values with multiple testing thresholds")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 5. Practical Guidance

| Scenario | Method | Why |
|---|---|---|
| 1–3 pre-specified primary metrics | None needed | Pre-registration handles it |
| Many confirmatory tests, costly false positives | Bonferroni or Holm | Strong FWER control |
| Exploratory segment mining | Benjamini-Hochberg | FDR is appropriate for discovery |
| Sequential peeking (multiple looks) | Alpha spending (O'Brien-Fleming) | Covered in extensions |

**The best defense against multiple testing problems is pre-registration:**
commit to your primary metric and planned segments *before* launch.
Post-hoc findings are exploratory, not confirmatory — label them accordingly.

Proceed to **[07 — Business Impact](07_business_impact.ipynb)**.